In [6]:
import os

import ollama
import chromadb
from langchain_community.document_loaders import PyPDFLoader

from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings

from utils.list_dir_in_dir import list_directories_in_directory
from utils.markdown_to_string import markdown_file_to_string
from utils.list_file_in_dir import list_files_in_directory
from utils.csv_to_markdown import csv_to_markdown
from utils.markdown_to_string import markdown_file_to_string

In [7]:
# ollama need to be created befoer runing 
embeddings = OllamaEmbeddings(model="hsbi")
import nest_asyncio
nest_asyncio.apply()
from llama_parse import LlamaParse
        
parser = LlamaParse(
    api_key="llx-VQsncrO19nFgA9cqAIxguyYViqbuc3XhxfwdRaUXm5Y4Mawf",  
    result_type="markdown",  # "markdown" and "text" are available
    num_workers=4,  # if multiple files passed, split in `num_workers` API calls
    verbose=True,
    language="en",  # Optionally you can define a language, default=en
)



        


In [8]:
# extract the dir

path = 'src'
ric_list = list_directories_in_directory(path)
print(ric_list)




['0002', '0011', '0038', '0069', '0136', '0151', '0271', '0806', '0909', '0933', '0975', '0981', '0990', '1055', '1182', '1310', '1347', '1398', '1456', '1477', '1658', '1681', '1787', '1833', '1836', '1880', '1909', '1929', '1988', '2013', '2020', '2145', '2162', '2273', '2362', '2416', '2469', '3383', '3668', '3759', '3888', '3900', '3918', '3958', '3969', '6889', '9636', '9858', '9860', '9923', '9930']


In [11]:
for ric in ric_list:   
        

        src_path = f'./src/{ric}'


        files = list_files_in_directory(src_path)
        
        
        csv_list = list()
        pdf_list = list()
        for file in files:
            file_name, file_extension = os.path.splitext(file)
            if file_extension.lower() == '.csv':
                csv_list.append(file)
            elif file_extension.lower() == '.pdf':
                pdf_list.append(file)
             
        # pdf 
        documents = parser.load_data([os.path.join(src_path, file) for file in pdf_list])
        text = [document.text for i, document in enumerate(documents)]
        print(text)
            
        # csv
        # convert to markdown for better understanding to llama3
        csv_data = list()
        for file in csv_list:
            file_name, file_extension = os.path.splitext(file)
            csv_to_markdown(os.path.join(src_path, file), os.path.join(src_path, file_name+'.md'))
            csv_data.append(markdown_file_to_string(os.path.join(src_path, file_name+'.md')))
            

        text = text + csv_data
    
        vector_directory = f'./temp/{ric}'
        os.makedirs(vector_directory, exist_ok=True)
        client = chromadb.PersistentClient(path=vector_directory)
        try:
            collection = client.create_collection(name=f"ric{ric}")
        except:
            collection = client.get_collection(name=f"ric{ric}")
        for i, d in enumerate(text):
            if d is None:
                continue
            response = ollama.embeddings(model="mxbai-embed-large", prompt=d)
            embedding = response["embedding"]
            collection.add(
                ids=[str(i)],
                embeddings=[embedding],
                documents=[d]
            )
        

Parsing files: 100%|██████████| 3/3 [03:38<00:00, 72.70s/it]


['# Foundations for a Sustainable Energy Future\n\n# Annual Report 2022\n\nStock Code: 00002', '# Welcome to CLP’s 2022 Annual Report\n\n2022 was another year of immense challenges for CLP and the markets that we operate in. While COVID-19 restrictions were on the wane, we were faced with the impact of an energy crisis and increased commodity prices resulting from the war in Ukraine. It is against this background that we wish to report to you, our stakeholders, on how we have been managing our business and on our financial and environmental, social and governance (ESG) performance in our Annual Report and Sustainability Report.\n\n# Foundations for a Sustainable Energy Future\n\n# 2022 Annual Report\n\nStock Code: 00002\n\nOur Annual Report covers and goes beyond discussing CLP’s financial performance. It articulates how we create value for our stakeholders, both financial and non-financial, i.e. our broader stakeholder groups. Through the double materiality methodology, the Annual Rep

Parsing files: 100%|██████████| 3/3 [02:31<00:00, 50.59s/it]


['YEA RS', "# CONTENTS\n\n|Title|Page|\n|---|---|\n|Corporate Profile|1|\n|Results in Brief|3|\n|Five-year Financial Summary|4|\n|Chairman’s Statement *|6|\n|Chief Executive’s Report *|9|\n|Management Discussion & Analysis| |\n|- Business Review|14|\n|- Financial Review|22|\n|- Risk|39|\n|Corporate Governance Report|147|\n|Biographical Details of Directors and Senior Management|188|\n|Report of the Directors|209|\n|2022 Financial Statements|216|\n|Independent Auditor's Report|296|\n|Analysis of Shareholders|303|\n|Subsidiaries|304|\n|Directors of Subsidiaries|305|\n|Corporate Information and Calendar|306|\n|Cautionary statement regarding forward-looking statements|309|\n\n* Where possible, percentages in this section have been rounded to the nearest percentage point to facilitate easy reading. Percentage-based indicators remain at 1 or 2 decimal places as appropriate.\n\nThe abbreviations ‘HK$m’ and ‘HK$bn’ represent millions and billions of Hong Kong dollars respectively.", '# CORPORA

Parsing files:  33%|███▎      | 1/3 [00:01<00:02,  1.48s/it]

...

Error while parsing the file './src/9930\2024010201448.pdf': Failed to parse the file: {"detail":"You've exceeded the maximum number of pages you can parse in a day (1000). Please contact support to increase your limit."}


Parsing files:  67%|██████▋   | 2/3 [00:27<00:14, 14.84s/it]

Error while parsing the file './src/9930\2023051200007.pdf': Failed to parse the file: {"detail":"You've exceeded the maximum number of pages you can parse in a day (1000). Please contact support to increase your limit."}


Parsing files: 100%|██████████| 3/3 [00:54<00:00, 18.11s/it]

Error while parsing the file './src/9930\2023090400063.pdf': Failed to parse the file: {"detail":"You've exceeded the maximum number of pages you can parse in a day (1000). Please contact support to increase your limit."}
[]
CSV file converted to Markdown successfully. Saved as ./src/9930\Complete list of directors.md
An error occurred: 'cp950' codec can't decode byte 0x95 in position 318: illegal multibyte sequence
CSV file converted to Markdown successfully. Saved as ./src/9930\Complete list of substantial shareholders.md
An error occurred: 'cp950' codec can't decode byte 0x95 in position 251: illegal multibyte sequence
CSV file converted to Markdown successfully. Saved as ./src/9930\Consolidated list of substantial shareholders.md
An error occurred: 'cp950' codec can't decode byte 0x95 in position 251: illegal multibyte sequence
CSV file converted to Markdown successfully. Saved as ./src/9930\List of all notices.md
An error occurred: 'cp950' codec can't decode byte 0x90 in position 

In [9]:



# ollama need to be created befoer runing 
for ric in ric_list:   
    prompt = f"What is the Number of issued shares (excluding treasury shares)of {ric}, balance at close of the month?(03380), which shouldn\'t is not with the authorized/registered shares"
    prompt_4_embedding_search = ''''# III. Details of Movements in Issued Shares and/or Treasury Shares
    '''

    vector_directory = f'./temp/{ric}'
    client = chromadb.PersistentClient(path=vector_directory)
    collection = client.get_collection(name=f"ric{ric}")
    # generate an embedding for the prompt and retrieve the most relevant doc
    response = ollama.embeddings(
      prompt=prompt_4_embedding_search,
      model="mxbai-embed-large"
    )
    print(response)
    # n_results is the number of page that the vector db take out
    results = collection.query(
      query_embeddings=[response["embedding"]],
      n_results=10
    )
    data = results['documents'][0][0]
    
    print(data)
    output = ollama.generate(
      model="hsbi",
      prompt=f"Using this data: {data}. Respond to this prompt: {prompt}"
    )
    
    print(output['response'])
        


{'embedding': [1.0361485481262207, 0.5807598829269409, 0.7780753970146179, 0.1258736401796341, -0.1389487236738205, -0.022003918886184692, -0.04404474422335625, 0.046299487352371216, 0.5600059032440186, 0.8230735063552856, 0.39655667543411255, 0.7141626477241516, -0.49878352880477905, 0.0958494022488594, -0.40975478291511536, 0.4020810127258301, -0.6986159086227417, -0.035263944417238235, -0.17925460636615753, -0.08047626912593842, -0.2873575687408447, 0.2652997374534607, -1.0807714462280273, 0.37529852986335754, -0.5668733716011047, 0.34830397367477417, -1.0234419107437134, 0.29417985677719116, 1.3012452125549316, 0.7091317772865295, -0.0961168184876442, -0.6223416328430176, 0.13980084657669067, -0.5443767309188843, 0.027376536279916763, 0.11752240359783173, -0.27419841289520264, -0.619850218296051, 0.4283481240272522, -0.4732462763786316, 0.04119753837585449, 0.3869864344596863, 0.5122289657592773, -0.11849614977836609, -1.228151559829712, -0.5245018005371094, -1.0388942956924438, -0

Number of requested results 10 is greater than number of elements in index 5, updating n_results = 5


{'embedding': [1.0361485481262207, 0.5807598829269409, 0.7780753970146179, 0.1258736401796341, -0.1389487236738205, -0.022003918886184692, -0.04404474422335625, 0.046299487352371216, 0.5600059032440186, 0.8230735063552856, 0.39655667543411255, 0.7141626477241516, -0.49878352880477905, 0.0958494022488594, -0.40975478291511536, 0.4020810127258301, -0.6986159086227417, -0.035263944417238235, -0.17925460636615753, -0.08047626912593842, -0.2873575687408447, 0.2652997374534607, -1.0807714462280273, 0.37529852986335754, -0.5668733716011047, 0.34830397367477417, -1.0234419107437134, 0.29417985677719116, 1.3012452125549316, 0.7091317772865295, -0.0961168184876442, -0.6223416328430176, 0.13980084657669067, -0.5443767309188843, 0.027376536279916763, 0.11752240359783173, -0.27419841289520264, -0.619850218296051, 0.4283481240272522, -0.4732462763786316, 0.04119753837585449, 0.3869864344596863, 0.5122289657592773, -0.11849614977836609, -1.228151559829712, -0.5245018005371094, -1.0388942956924438, -0

Number of requested results 10 is greater than number of elements in index 6, updating n_results = 6


{'embedding': [1.0361485481262207, 0.5807598829269409, 0.7780753970146179, 0.1258736401796341, -0.1389487236738205, -0.022003918886184692, -0.04404474422335625, 0.046299487352371216, 0.5600059032440186, 0.8230735063552856, 0.39655667543411255, 0.7141626477241516, -0.49878352880477905, 0.0958494022488594, -0.40975478291511536, 0.4020810127258301, -0.6986159086227417, -0.035263944417238235, -0.17925460636615753, -0.08047626912593842, -0.2873575687408447, 0.2652997374534607, -1.0807714462280273, 0.37529852986335754, -0.5668733716011047, 0.34830397367477417, -1.0234419107437134, 0.29417985677719116, 1.3012452125549316, 0.7091317772865295, -0.0961168184876442, -0.6223416328430176, 0.13980084657669067, -0.5443767309188843, 0.027376536279916763, 0.11752240359783173, -0.27419841289520264, -0.619850218296051, 0.4283481240272522, -0.4732462763786316, 0.04119753837585449, 0.3869864344596863, 0.5122289657592773, -0.11849614977836609, -1.228151559829712, -0.5245018005371094, -1.0388942956924438, -0

KeyboardInterrupt: 